# ReconMind Stage 3, Model B: fine tuned SciBERT ATT&CK mapper (training run 2)

This notebook teaches a small language model (SciBERT) to read one sentence from a threat report and say which MITRE ATT&CK techniques it describes. It is Model B of the experiment. Model A is a prompted LLM scored by `evaluation/scripts/eval_prompted.py`.

To keep the comparison fair, Model B trains only on the train split, makes every choice (best epoch, threshold) on the validation split, and is scored once on the exact same eval set as Model A with the same scoring code.

**What changed since run 1.** Run 1 trained for 5 epochs at a constant learning rate of 2e-5 with plain BCE loss. It reached validation micro F1 0.435 and was still getting better every epoch, so it stopped too soon. It also rarely predicted the rare techniques (low recall). Run 2 changes four things and nothing else:

1. **More epochs, with early stopping.** Up to 15 epochs instead of 5, but training stops once validation micro F1 has not improved for 3 epochs in a row. The best epoch's weights are still the ones kept.
2. **A learning rate schedule.** The learning rate climbs from zero to 3e-5 over the first 10 percent of training steps (warmup), then falls in a straight line back to zero by the last planned step.
3. **Extra weight on rare techniques.** Missing a rare technique now costs more in the loss, so the model cannot score well by ignoring it.
4. **Run names on every output.** Results and the saved model carry the name `run2`, so run 1's files are never overwritten.

The seed, the eval fingerprint check, the plain SciBERT starting point, choosing the epoch and threshold on validation only, predicting the eval set once, and the scoring code are all unchanged.

**How to run:** use a T4 GPU (section 2 checks), then choose Runtime, Run all. Run 1 took about 2 minutes 15 seconds per epoch, so allow up to about 45 minutes if all 15 epochs run. Near the end Google asks for permission to use your Drive, where the trained model is saved.

The notebook clones the `attack-mapper` branch from GitHub, so that branch must already contain `evaluation/attack_mapper/finetune_utils.py` with `compute_pos_weight` in it.

## 1. Run settings

Everything that differs from run 1 is set here, in one place, and every value is written to the metrics file so the run can be described exactly later.

- `RUN_NAME` goes into the name of every output file and the Drive folder.
- `EPOCHS` is the most passes over the training data. Early stopping may end training sooner.
- `PATIENCE` is how many epochs in a row may pass without a better validation micro F1 before training stops.
- `LEARNING_RATE` is the peak learning rate, reached at the end of warmup.
- `WARMUP_FRACTION` is the share of all planned training steps spent warming up.
- `POS_WEIGHT_CAP` is the largest extra weight any single technique can get in the loss.

In [ ]:
RUN_NAME = 'run2'
EPOCHS = 15
PATIENCE = 3
LEARNING_RATE = 3e-5
WARMUP_FRACTION = 0.1
POS_WEIGHT_CAP = 30.0

assert RUN_NAME and RUN_NAME != 'run1', 'choose a new RUN_NAME so the run 1 results are kept'
print('run:', RUN_NAME, '| epochs up to', EPOCHS, '| patience', PATIENCE,
      '| lr', LEARNING_RATE, '| warmup', WARMUP_FRACTION, '| pos_weight cap', POS_WEIGHT_CAP)

## 2. Check for a GPU

Training on a CPU would take many hours, so this cell stops right away if no GPU is attached and tells you how to add one.

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        'No GPU is attached. In the Colab menu choose Runtime, Change runtime type, '
        'pick T4 GPU, click Save, then choose Runtime, Run all.')

GPU_NAME = torch.cuda.get_device_name(0)
print('GPU:', GPU_NAME)
if 'T4' not in GPU_NAME:
    print('Note: this notebook was written for a T4. Other GPUs work, timings will differ.')

## 3. Get the ReconMind code

Downloads the project from GitHub (branch `attack-mapper`) so we can reuse its data preparation and scoring code instead of copying it here. The commit ID is recorded so the result can be traced to the exact code that produced it.

In [ ]:
import os
import subprocess
import sys

REPO_URL = 'https://github.com/r00tSud0/ReconMind'
BRANCH = 'attack-mapper'
REPO_DIR = '/content/ReconMind'

if os.path.isdir(REPO_DIR):
    print('Already cloned, reusing', REPO_DIR)
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--depth', '1', REPO_URL, REPO_DIR],
                   check=True)

os.chdir(REPO_DIR)
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

CODE_COMMIT = subprocess.run(['git', 'rev-parse', 'HEAD'], check=True,
                             capture_output=True, text=True).stdout.strip()
print('code commit:', CODE_COMMIT)

## 4. Install the Hugging Face library

`transformers` provides SciBERT, its tokenizer and the learning rate schedule. PyTorch already comes with Colab. The versions are printed and saved with the results.

In [ ]:
!pip install -q transformers

import transformers

print('torch', torch.__version__, '| transformers', transformers.__version__)

## 5. Prepare the TRAM data

Runs the same script Model A used. It downloads MITRE's TRAM sentences pinned to one commit, checks the file's checksum, and writes the train, validation, test and eval splits to `evaluation/data/tram/`. If anything is wrong it fails here and Run all stops.

In [ ]:
subprocess.run([sys.executable, 'evaluation/scripts/prepare_tram_data.py'], check=True)

## 6. Confirm the eval set is the one Model A was scored on

A fingerprint (SHA-256) of `eval.jsonl` is computed exactly the way `evaluation/scripts/eval_prompted.py` computes `eval_sha256`: over the raw file bytes. If it does not match the known value, the two models would not be comparable, so the notebook stops.

In [ ]:
import hashlib

EXPECTED_EVAL_SHA256 = '46860e84293aaccc1d7fdc1094e036dd4c4d4a717d432eb6493cbe08eb20445c'
DATA_DIR = os.path.join(REPO_DIR, 'evaluation', 'data', 'tram')

# Same steps as _load_eval in evaluation/scripts/eval_prompted.py.
with open(os.path.join(DATA_DIR, 'eval.jsonl'), 'rb') as fh:
    eval_raw = fh.read()
EVAL_SHA256 = hashlib.sha256(eval_raw).hexdigest()

print('eval fingerprint:', EVAL_SHA256)
if EVAL_SHA256 != EXPECTED_EVAL_SHA256:
    raise RuntimeError('The eval set fingerprint does not match the one Model A used '
                       '(expected ' + EXPECTED_EVAL_SHA256 + '). Stopping so the '
                       'results stay comparable.')
print('OK: same eval set as Model A.')

## 7. Load the sentences and the 50 labels

Reads the splits and the list of 50 technique IDs. Each sentence's labels become a row of 50 zeros and ones (a "multi-hot" vector), which is the target the model learns to predict. A sentence can have several techniques, or none.

In [ ]:
import json

from evaluation.attack_mapper import finetune_utils, metrics


def read_jsonl(name):
    with open(os.path.join(DATA_DIR, name)) as fh:
        return [json.loads(line) for line in fh if line.strip()]


train_rows = read_jsonl('train.jsonl')
val_rows = read_jsonl('val.jsonl')
eval_rows = [json.loads(line) for line in eval_raw.decode('utf-8').splitlines() if line.strip()]

with open(os.path.join(DATA_DIR, 'labels.json')) as fh:
    LABELS = [t['id'] for t in json.load(fh)]
assert len(LABELS) == 50, 'expected 50 labels, found {n}'.format(n=len(LABELS))

for name, rows in (('train', train_rows), ('val', val_rows), ('eval', eval_rows)):
    labeled = sum(1 for r in rows if r['labels'])
    print('{n:<5} {s:>6} sentences, {l:>5} with at least one technique'.format(
        n=name, s=len(rows), l=labeled))

## 8. Fix the random seed

Setting seed 42 everywhere makes the run repeatable: the same shuffling and the same starting weights for the new output layer. This is the same seed as run 1, so the differences between the runs come from the new settings, not from luck. Training in half precision (fp16) on a GPU is close to, but not bit for bit, reproducible, so a rerun may differ in the last decimals.

In [ ]:
import random

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
transformers.set_seed(SEED)
print('seed', SEED)

## 9. Load the tokenizer and pick the maximum sentence length

We start from plain SciBERT (`allenai/scibert_scivocab_uncased`), a general science model, and never from MITRE's own TRAM fine tuned checkpoint, which has already seen this data. Run 2 starts from plain SciBERT again, not from run 1's model.

The tokenizer splits sentences into tokens. Longer limits are slower, so we use 256 tokens unless more than 1 percent of training sentences would be cut off, in which case we use 512.

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'allenai/scibert_scivocab_uncased'
assert 'tram' not in MODEL_NAME.lower(), 'start from plain SciBERT, not a TRAM checkpoint'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Untruncated lengths, counting the [CLS] and [SEP] tokens the model adds.
train_lengths = [len(ids) for ids in tokenizer([r['sentence'] for r in train_rows])['input_ids']]
MAX_LENGTH, CUT_AT_256 = finetune_utils.choose_max_length(train_lengths)

print('{p:.2f}% of training sentences are longer than 256 tokens'.format(p=100 * CUT_AT_256))
if MAX_LENGTH == 512:
    cut_512 = sum(1 for n in train_lengths if n > 512) / len(train_lengths)
    print('{p:.2f}% are longer than 512 tokens'.format(p=100 * cut_512))
print('max length:', MAX_LENGTH)

## 10. Turn the sentences into batches

Tokenizes every sentence once and groups them into batches. Training batches hold 16 sentences and are shuffled (with the fixed seed). Validation and eval batches are larger and keep their order, since no learning happens there. Each batch is padded only to its own longest sentence, which saves time.

The training set is kept in its own variable (`train_set`) because section 11 counts its labels to set the rare technique weights.

In [ ]:
from torch.utils.data import DataLoader, Dataset
from transformers import DataCollatorWithPadding

BATCH_SIZE = 16
EVAL_BATCH_SIZE = 64


class SentenceSet(Dataset):
    def __init__(self, rows):
        self.enc = tokenizer([r['sentence'] for r in rows], truncation=True,
                             max_length=MAX_LENGTH)
        self.targets = [finetune_utils.to_multi_hot(r['labels'], LABELS) for r in rows]

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.enc.items()}
        item['labels'] = self.targets[i]
        return item


pad = DataCollatorWithPadding(tokenizer)


def collate(items):
    labels = torch.tensor([item.pop('labels') for item in items], dtype=torch.float)
    batch = pad(items)
    batch['labels'] = labels
    return batch


train_set = SentenceSet(train_rows)
shuffle_gen = torch.Generator()
shuffle_gen.manual_seed(SEED)
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          generator=shuffle_gen, collate_fn=collate)
val_loader = DataLoader(SentenceSet(val_rows), batch_size=EVAL_BATCH_SIZE, collate_fn=collate)
eval_loader = DataLoader(SentenceSet(eval_rows), batch_size=EVAL_BATCH_SIZE, collate_fn=collate)
print('training batches per epoch:', len(train_loader))

## 11. Load SciBERT with a 50 label output layer and a weighted loss

Adds a fresh output layer with one score per technique. Each score is judged on its own (yes or no), which is what "multi label" means. The loss is binary cross entropy on the raw scores (BCE with logits). A warning that some weights are "newly initialized" is expected: that is the new output layer.

**New in run 2: extra weight for rare techniques.** Most techniques appear in only a few training sentences. With plain BCE the model can get a low loss by almost never predicting them, which is why run 1's recall on rare techniques was low. Now each technique gets its own weight (`pos_weight`) equal to the number of training sentences without it divided by the number with it. A technique found in 1 sentence out of 100 gets weight 99, so missing it costs about as much as all 99 correct "no" answers put together. Common techniques get a small weight.

Two safety rules:

- The weights are capped at `POS_WEIGHT_CAP` (30), so a technique seen only once or twice cannot take over training. A technique with no training examples at all also gets the cap.
- The counts come from the train split only. Counting validation or eval sentences would leak information about them into training.

Because of the weights, the training loss numbers in run 2 are larger than run 1's and cannot be compared with them. Compare runs on validation F1 instead.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    problem_type='multi_label_classification',
    id2label=dict(enumerate(LABELS)),
    label2id={lab: i for i, lab in enumerate(LABELS)},
).to('cuda')

# negatives / positives per technique, counted on the TRAIN split only, capped.
POS_WEIGHT = finetune_utils.compute_pos_weight(train_set.targets, POS_WEIGHT_CAP)
loss_fn = torch.nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(POS_WEIGHT, dtype=torch.float, device='cuda'))

n_capped = sum(1 for w in POS_WEIGHT if w >= POS_WEIGHT_CAP)
print('pos_weight: min {lo:.2f}, median {md:.2f}, max {hi:.2f}'.format(
    lo=min(POS_WEIGHT), md=float(np.median(POS_WEIGHT)), hi=max(POS_WEIGHT)))
print('{n} of {t} techniques are at the cap of {c}'.format(
    n=n_capped, t=len(LABELS), c=POS_WEIGHT_CAP))

## 12. A helper that turns sentences into probabilities

Runs the model without learning and squashes each of the 50 scores into a probability between 0 and 1 (the sigmoid function). Used for validation and, once, for the eval set.

In [ ]:
@torch.no_grad()
def predict_probs(loader):
    model.eval()
    probs = []
    for batch in loader:
        inputs = {k: v.to('cuda') for k, v in batch.items() if k != 'labels'}
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(**inputs).logits
        probs.extend(torch.sigmoid(logits.float()).cpu().tolist())
    return probs

## 13. Train

Up to `EPOCHS` (15) passes over the training sentences with the AdamW optimizer, in half precision (fp16) for speed. After each epoch the model is checked on the validation split only: we try thresholds from 0.05 to 0.95 and take the best micro F1. The epoch with the best validation score is kept (an earlier epoch wins a tie). The eval set is not touched here.

**New in run 2: learning rate schedule.** Run 1 used the same learning rate from the first step to the last. Run 2 starts at zero and climbs in a straight line to the peak of 3e-5 over the first 10 percent of all planned training steps. This warmup keeps the large, early updates from damaging what SciBERT already knows while its new output layer is still random. After the peak the rate falls in a straight line, reaching zero at the end of epoch 15, so late epochs make small, careful adjustments. The schedule is planned for all 15 epochs; if early stopping ends training sooner, the rate simply never reaches zero.

**New in run 2: early stopping.** Run 1 was still improving when it hit its 5 epoch limit. Run 2 allows up to 15 epochs, but stops once validation micro F1 has failed to beat its best so far for `PATIENCE` (3) epochs in a row. A tie with the best does not count as an improvement. Stopping does not change which weights are kept: the best epoch's weights are saved as training goes and restored in section 14.

The learning rate at the end of each epoch is printed and saved with the epoch history.

In [ ]:
import time

from transformers import get_linear_schedule_with_warmup

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
TOTAL_STEPS = EPOCHS * len(train_loader)
WARMUP_STEPS = int(WARMUP_FRACTION * TOTAL_STEPS)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=TOTAL_STEPS)
scaler = torch.amp.GradScaler('cuda')
val_gold = [r['labels'] for r in val_rows]
print('{t} training steps planned, the first {w} are warmup'.format(
    t=TOTAL_STEPS, w=WARMUP_STEPS))

best_f1, best_epoch, best_state = -1.0, None, None
epochs_since_best = 0
STOPPED_EARLY = False
epoch_history = []
train_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    epoch_start = time.time()
    loss_sum, steps = 0.0, 0
    for batch in train_loader:
        batch = {k: v.to('cuda') for k, v in batch.items()}
        targets = batch.pop('labels')
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            logits = model(**batch).logits
        loss = loss_fn(logits.float(), targets)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        loss_sum += loss.item()
        steps += 1

    val_probs = predict_probs(val_loader)
    threshold, val_f1, _ = finetune_utils.sweep_threshold(val_gold, val_probs, LABELS)
    val_f1_at_half = metrics.score(
        val_gold, finetune_utils.predict_all(val_probs, LABELS, 0.5), LABELS)['micro']['f1']
    lr_now = scheduler.get_last_lr()[0]
    row = {'epoch': epoch, 'train_loss': loss_sum / steps, 'val_micro_f1': val_f1,
           'val_threshold': threshold, 'val_micro_f1_at_0.5': val_f1_at_half,
           'lr_at_epoch_end': lr_now, 'seconds': round(time.time() - epoch_start, 1)}
    epoch_history.append(row)

    kept = val_f1 > best_f1
    if kept:
        best_f1, best_epoch = val_f1, epoch
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        epochs_since_best = 0
    else:
        epochs_since_best += 1
    print('epoch {e}: loss {l:.4f}  val micro F1 {f:.3f} at threshold {t}  '
          '(F1 at 0.5: {h:.3f})  lr {r:.2e}  {s:.0f}s{k}'.format(
              e=epoch, l=row['train_loss'], f=val_f1, t=threshold, h=val_f1_at_half,
              r=lr_now, s=row['seconds'], k='  <- best so far' if kept else ''))

    if epochs_since_best >= PATIENCE:
        STOPPED_EARLY = True
        print('No validation improvement for {p} epochs, stopping early.'.format(p=PATIENCE))
        break

EPOCHS_RUN = len(epoch_history)
TRAIN_SECONDS = round(time.time() - train_start, 1)
print('epochs run: {n} of {m} | best epoch: {b} | training time: {t:.1f} min'.format(
    n=EPOCHS_RUN, m=EPOCHS, b=best_epoch, t=TRAIN_SECONDS / 60))

## 14. Restore the best epoch and pick the threshold

Loads the weights from the best epoch back into the model, which matters most when early stopping ran a few epochs past it. A technique is predicted when its probability is at least the threshold, and one threshold is used for all 50 techniques. It is chosen on validation only (best micro F1, lowest threshold on ties), never on the eval set.

In [ ]:
model.load_state_dict(best_state)
val_probs = predict_probs(val_loader)
THRESHOLD, VAL_F1, val_sweep = finetune_utils.sweep_threshold(val_gold, val_probs, LABELS)

print('threshold   P      R      F1  (validation)')
for row in val_sweep:
    print('  {t:.2f}    {p:.3f}  {r:.3f}  {f:.3f}{m}'.format(
        t=row['threshold'], p=row['precision'], r=row['recall'], f=row['f1'],
        m='  <- chosen' if row['threshold'] == THRESHOLD else ''))

## 15. Score the eval set, once

Predicts the eval set a single time with the chosen epoch and threshold, then scores it with `evaluation/attack_mapper/metrics.py`, the same code that scored Model A.

In [ ]:
eval_probs = predict_probs(eval_loader)
eval_preds = finetune_utils.predict_all(eval_probs, LABELS, THRESHOLD)

gold_by_id = {r['id']: r['labels'] for r in eval_rows}
pred_by_id = {r['id']: p for r, p in zip(eval_rows, eval_preds)}
result = metrics.score(gold_by_id, pred_by_id, LABELS)

mi, ma = result['micro'], result['macro']
print('-' * 60)
print('sentences:    {n} ({l} labeled, {u} unlabeled)'.format(
    n=result['n_sentences'], l=result['n_labeled'], u=result['n_unlabeled']))
print('micro:        P {p:.3f}  R {r:.3f}  F1 {f:.3f}'.format(
    p=mi['precision'], r=mi['recall'], f=mi['f1']))
print('macro F1:     {f:.3f} over {n} techniques'.format(
    f=ma['f1'], n=ma['n_techniques_averaged']))
print('exact match:  {e:.3f}'.format(e=result['exact_match']))
print('out of set:   {n} predicted IDs'.format(n=result['out_of_set']['count']))
if result['zero_support_techniques']:
    print('zero support: {z}'.format(z=', '.join(result['zero_support_techniques'])))

## 16. Save the results

Writes `evaluation/results/finetuned_scibert_run2_metrics.json` in the same layout as Model A's metrics file, plus the training details. Besides what run 1 recorded (best epoch, threshold, max length, time, seed, library versions), run 2 also records every run setting from section 1, how many epochs actually ran and whether early stopping ended training, the warmup and total step counts, and the `pos_weight` given to each technique.

Also writes one line per eval sentence to `evaluation/results/raw/finetuned_scibert_run2.jsonl` with the gold labels, the predicted labels and all 50 probabilities, so the threshold can be rechecked later without retraining.

Both file names come from `RUN_NAME`, so run 1's files are never overwritten.

In [ ]:
from datetime import datetime, timezone

RESULTS_DIR = os.path.join(REPO_DIR, 'evaluation', 'results')
os.makedirs(os.path.join(RESULTS_DIR, 'raw'), exist_ok=True)
METRICS_PATH = os.path.join(RESULTS_DIR, 'finetuned_scibert_{r}_metrics.json'.format(r=RUN_NAME))
RAW_PATH = os.path.join(RESULTS_DIR, 'raw', 'finetuned_scibert_{r}.jsonl'.format(r=RUN_NAME))

out = {
    'model_kind': 'finetuned',
    'provider': 'huggingface',
    'model': MODEL_NAME,
    'run_name': RUN_NAME,
    'batch_size': BATCH_SIZE,
    'eval_sha256': EVAL_SHA256,
    'failed_sentences': 0,
    'written': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'epochs': EPOCHS,
    'epochs_run': EPOCHS_RUN,
    'stopped_early': STOPPED_EARLY,
    'patience': PATIENCE,
    'best_epoch': best_epoch,
    'best_val_micro_f1': VAL_F1,
    'threshold': THRESHOLD,
    'max_length': MAX_LENGTH,
    'pct_truncated_at_256': round(100 * CUT_AT_256, 3),
    'train_seconds': TRAIN_SECONDS,
    'seed': SEED,
    'learning_rate': LEARNING_RATE,
    'lr_schedule': 'linear warmup then linear decay to zero',
    'warmup_fraction': WARMUP_FRACTION,
    'warmup_steps': WARMUP_STEPS,
    'total_steps': TOTAL_STEPS,
    'loss': 'BCEWithLogitsLoss with pos_weight',
    'pos_weight_cap': POS_WEIGHT_CAP,
    'pos_weight_counted_on': 'train',
    'pos_weight': {lab: round(w, 4) for lab, w in zip(LABELS, POS_WEIGHT)},
    'fp16': True,
    'torch_version': torch.__version__,
    'transformers_version': transformers.__version__,
    'gpu': GPU_NAME,
    'code_commit': CODE_COMMIT,
    'epoch_history': epoch_history,
    'val_threshold_sweep': val_sweep,
    'metrics': result,
}
with open(METRICS_PATH, 'w') as fh:
    json.dump(out, fh, indent=1)
    fh.write('\n')

with open(RAW_PATH, 'w') as fh:
    for r, pred, probs in zip(eval_rows, eval_preds, eval_probs):
        fh.write(json.dumps({'id': r['id'], 'gold': r['labels'], 'pred': pred,
                             'probs': [round(p, 4) for p in probs]}, sort_keys=True) + '\n')

print('metrics:', METRICS_PATH)
print('raw:    ', RAW_PATH)

## 17. Save the model to Google Drive

Colab deletes everything when the session ends, so the trained model and tokenizer are saved to `MyDrive/ReconMind/scibert_attack_mapper_run2`. This is a new folder named from `RUN_NAME`; run 1's model in `MyDrive/ReconMind/scibert_attack_mapper` is left alone. A popup asks you to allow access to your Drive. Copies of `labels.json` and both result files go to the same folder.

In [ ]:
import shutil

from google.colab import drive

drive.mount('/content/drive')
SAVE_DIR = '/content/drive/MyDrive/ReconMind/scibert_attack_mapper_{r}'.format(r=RUN_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

model.save_pretrained(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
for path in (os.path.join(DATA_DIR, 'labels.json'), METRICS_PATH, RAW_PATH):
    shutil.copy(path, SAVE_DIR)
print('saved to', SAVE_DIR)

## 18. Download the metrics file

Offers `finetuned_scibert_run2_metrics.json` as a download. Put it in the `evaluation/results/` folder of your local ReconMind copy and commit it next to Model A's and run 1's metrics files. (The per sentence file is in your Drive folder; `evaluation/results/raw/` is not committed.)

In [ ]:
from google.colab import files

files.download(METRICS_PATH)